In [2]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import itertools
from tqdm import tqdm
plt.style.use("ggplot")

def build_cnn(input_shape, num_classes):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(tf.keras.layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(tf.keras.layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(512, activation='relu'))
    model.add(tf.keras.layers.Dense(num_classes, activation='softmax'))
    model.compile(loss='sparse_categorical_crossentropy', 
                  optimizer=tf.keras.optimizers.Adam(), 
                  metrics=['accuracy'])
    print(model.summary())
    return model

def train_cnn(model, x_train, y_train, x_val, y_val, epochs, batch_size):
    history = model.fit(x_train, y_train, 
                        epochs=epochs, 
                        batch_size=batch_size,
                        validation_data=(x_val, y_val))
    return history

def plot_loss(history, epochs, title):
    plt.figure(figsize=(12, 8))
    plt.plot(np.arange(0, epochs), history.history["loss"], label="train_loss")
    plt.plot(np.arange(0, epochs), history.history["val_loss"], label="val_loss")
    plt.plot(np.arange(0, epochs), history.history["accuracy"], label="train_acc")
    plt.plot(np.arange(0, epochs), history.history["val_accuracy"], label="val_acc")
    plt.title(f"Training Loss and Accuracy - {title}")
    plt.xlabel("Epoch #")
    plt.ylabel("Loss/Accuracy")
    plt.legend()
    plt.show()

def load_img_data(size, df):
    img_h, img_w = size, size
    imgs = []
    image_paths = list(df['path'])

    for i in tqdm(range(len(image_paths))):
        img = cv2.imread(image_paths[i])
        img = cv2.resize(img, (img_h, img_w))
        img = img.astype(np.float32) / 255.
        imgs.append(img)

    imgs = np.stack(imgs, axis=0)
    print(imgs.shape)
    
    return imgs, df['cell_type_idx'].values

def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix', 
                          cmap=plt.cm.Blues):
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

if __name__ == "__main__":
    base_skin_dir = os.path.join('data', 'images')
    lesion_type_dict = {
        'nv': 'Melanocytic nevi',
        'mel': 'Melanoma',
        'bkl': 'Benign keratosis-like lesions',
        'bcc': 'Basal cell carcinoma',
        'akiec': 'Actinic keratoses',
        'vasc': 'Vascular lesions',
        'df': 'Dermatofibroma'
    }

    data = pd.read_csv('data/HAM10000_metadata.csv')
    data['path'] = "data/images/" + data['image_id'] + ".jpg"
    data['cell_type'] = data['dx'].map(lesion_type_dict.get) 
    data['cell_type_idx'] = pd.Categorical(data['cell_type']).codes

    imgs, target = load_img_data(128, data)
    x_train, x_test, y_train, y_test = train_test_split(imgs, target, test_size=0.20)
    x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.05)

    input_shape = x_train[0].shape
    num_classes = len(set(target))
    batch_size = 128
    epochs = 100

    model = build_cnn(input_shape, num_classes)
    history = train_cnn(model, x_train, y_train, x_val, y_val, epochs, batch_size)

    plot_loss(history, epochs, 'Simple CNN')

    test_loss, test_acc = model.evaluate(x_test, y_test)
    print(f"Test accuracy: {test_acc}")

    y_pred = model.predict(x_test)
    y_pred_classes = np.argmax(y_pred, axis=1)

    confusion_mtx = confusion_matrix(y_test, y_pred_classes)
    plot_confusion_matrix(confusion_mtx, classes=list(lesion_type_dict.values()))    


100%|██████████| 10015/10015 [01:55<00:00, 86.64it/s]


(10015, 128, 128, 3)
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 126, 126, 32)      896       
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 63, 63, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_4 (Conv2D)           (None, 61, 61, 64)        18496     
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 30, 30, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 28, 28, 128)       73856     
                                                                 
 max_pooling2d_5 (MaxPoolin  (Non

KeyboardInterrupt: 

In [4]:
model.save('skin_lesion_model.h5')

c:\Users\sujay\anaconda3\envs\machinelearning\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [3]:
import cv2
import numpy as np
import socket
from tensorflow.keras.models import load_model
from IPython.display import display, Image as IPImage
from PIL import Image
import io
import time


In [4]:
# Load the pre-trained model (assuming it's in the same directory)
model = load_model('skin_lesion_model.h5')

# Lesion type mapping
lesion_type_dict = {
    0: 'Melanocytic nevi',
    1: 'Melanoma',
    2: 'Benign keratosis-like lesions',
    3: 'Basal cell carcinoma',
    4: 'Actinic keratoses',
    5: 'Vascular lesions',
    6: 'Dermatofibroma'
}



In [5]:
# Function to preprocess the image for model input
def preprocess_image(image, size=128):
    img = cv2.resize(image, (size, size))  # Resize to 128x128
    img = img.astype(np.float32) / 255.0  # Normalize to [0, 1]
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    return img

# Function to classify the image using the model
def classify_image(model, preprocessed_image):
    predictions = model.predict(preprocessed_image)
    predicted_class_idx = np.argmax(predictions, axis=1)
    predicted_class = lesion_type_dict[predicted_class_idx[0]]
    return predicted_class


In [6]:
# UDP socket setup
UDP_IP = "0.0.0.0"  # Listen on all available interfaces
UDP_PORT = 9999  # The port you're listening on

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind((UDP_IP, UDP_PORT))  # Bind to all interfaces
print(f"Listening on {UDP_IP}:{UDP_PORT}")


Listening on 0.0.0.0:9999


In [8]:
# Function to capture, classify and display one frame at a time
def receive_and_classify():
    # Receive data from the UDP socket
    data, addr = sock.recvfrom(65536)  # Adjust buffer size if necessary
    nparr = np.frombuffer(data, np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)  # Decode to image format
    
    if frame is not None:
        # Preprocess the image for model input
        preprocessed_image = preprocess_image(frame)

        # Classify the image
        predicted_class = classify_image(model, preprocessed_image)

        # Convert OpenCV image to PIL format for display in Jupyter
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        # Display the frame and the predicted class
        display(image)
        print(f"Predicted Class: {predicted_class}")


In [9]:
# Example of receiving and classifying 10 frames
for i in range(10):
    receive_and_classify()
    time.sleep(0.5)  # Add a delay if you want to slow down the frame capture


In [1]:
import cv2
import numpy as np
import socket
from tensorflow.keras.models import load_model
from IPython.display import display, Image as IPImage
from PIL import Image
import io
import time

# Load your pre-trained model (make sure the model file is accessible)
model = load_model('skin_lesion_model.h5')

# Lesion type mapping
lesion_type_dict = {
    0: 'Melanocytic nevi',
    1: 'Melanoma',
    2: 'Benign keratosis-like lesions',
    3: 'Basal cell carcinoma',
    4: 'Actinic keratoses',
    5: 'Vascular lesions',
    6: 'Dermatofibroma'
}

# Preprocess the image for model input
def preprocess_image(image, size=128):
    img = cv2.resize(image, (size, size))  # Resize to 128x128
    img = img.astype(np.float32) / 255.0  # Normalize to [0, 1]
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    return img

# Classify the image using the model
def classify_image(model, preprocessed_image):
    predictions = model.predict(preprocessed_image)
    predicted_class_idx = np.argmax(predictions, axis=1)
    predicted_class = lesion_type_dict[predicted_class_idx[0]]
    return predicted_class

# Setup the UDP socket to receive data
UDP_IP = "0.0.0.0"  # Listen on all interfaces
UDP_PORT = 9999     # Port that the Raspberry Pi is sending to

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind((UDP_IP, UDP_PORT))  # Bind to all interfaces
print(f"Listening on {UDP_IP}:{UDP_PORT}")

# Receive, classify, and display the frame
def receive_and_classify_debug():
    # Receive data from the UDP socket
    data, addr = sock.recvfrom(65536)  # Buffer size
    nparr = np.frombuffer(data, np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)  # Decode to image format

    if frame is not None:
        print("Frame received and decoded successfully")
        print("Frame shape:", frame.shape)

        # Preprocess the image for model input
        preprocessed_image = preprocess_image(frame)

        # Classify the image
        try:
            predicted_class = classify_image(model, preprocessed_image)
            print(f"Predicted Class: {predicted_class}")
        except Exception as e:
            print(f"Error in classification: {e}")
        
        # Convert OpenCV image to PIL format for display in Jupyter
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        display(image)
    else:
        print("Failed to decode the frame")

# Call this function to test receiving and classifying one frame
receive_and_classify_debug()


Listening on 0.0.0.0:9999


In [ ]:
import cv2, socket, pickle, os    # Import Modules

# AF_INET refers to the address of family of ip4v
# SOCK_DGRAM means connection oriented UDP protocol
s=socket.socket(socket.AF_INET , socket.SOCK_DGRAM)  # Gives UDP protocol to follow
s.setsockopt(socket.SOL_SOCKET, socket.SO_SNDBUF, 10000000) # setSockoptTo open two protocols,SOL_SOCKET: Request applies to socket layer.
serverip="192.168..110.74"       # Server public IP
serverport=2323                # Server Port Number to identify the process that needs to recieve or send packets

cap = cv2.VideoCapture(0)       # Start Streaming video, will return video from your first webcam

# In order to iterate over block of code as long as test expression is true
while True:
    ret,photo = cap.read()      # Start Capturing a images/video
    cv2.imshow('my pic', photo) # Show Video/Stream
    ret, buffer = cv2.imencode(".jpg", photo, [int(cv2.IMWRITE_JPEG_QUALITY),30])  # ret will returns whether connected or not, Encode image from image to Buffer code(like [123,123,432....])
    x_as_bytes = pickle.dumps(buffer)       # Convert normal buffer Code(like [123,123,432....]) to Byte code(like b"\x00lOCI\xf6\xd4...")
    s.sendto(x_as_bytes,(serverip , serverport)) # Converted byte code is sending to server(serverip:serverport)
    if cv2.waitKey(10) == 13:    # Press Enter then window will close
        break                    
# Destroy all Windows/close
cv2.destroyAllWindows() 
cap.release()

In [2]:
import cv2, socket, numpy, pickle    # Import Modules

# AF_INET refers to the address of family of ip4v
# SOCK_DGRAM means connection oriented UDP protocol
s=socket.socket(socket.AF_INET , socket.SOCK_DGRAM)  # Gives UDP protocol to follow
ip="192.168.110.56"   # Server public IP
port=2323             # Server Port Number to identify the process that needs to recieve or send packets
s.bind(('0.0.0.0', port))  # Bind the IP:port to connect 

# In order to iterate over block of code as long as test expression is true
while True:
    x=s.recvfrom(100000000)    # Recieve byte code sent by client using recvfrom
    clientip = x[1][0]         # x[1][0] in this client details stored,x[0][0] Client message Stored
    data=x[0]                  # Data sent by client
    data=pickle.loads(data)    # All byte code is converted to Numpy Code 
    data = cv2.imdecode(data, cv2.IMREAD_COLOR)  # Decode 
    cv2.imshow('my pic', data) # Show Video/Stream
    if cv2.waitKey(10) == 13:  # Press Enter then window will close
        break
cv2.destroyAllWindows()        # Close all windows